# 03. 저랭크(LoRA) 한국어 추가학습 (A100 40GB, 약 1~2시간 / 약 6~11유닛)
`ko-reap-k16`(전문가 16개) 모델에 한국어 지시문(KULLM-v2 6천 개)으로 LoRA를 학습합니다.
- 어텐션 q/k/v/o + 남은 전문가 가중치에 LoRA, 라우터는 고정
- 학습 후 합친 모델을 GGUF로 바꾸고 다시 4비트(MXFP4_MOE)로 양자화 → 드라이브에 저장
- 평가는 04 노트북(L4)에서 합니다 (더 싸서)

In [ ]:
# 저장소 받기 + 패키지 설치
BRANCH = "ccr-a463bea6-bpbyrh"   # main에 합친 뒤에는 "main"
REPO = "github.com/b1bibig/Cfinetune"
import os
try:  # 저장소가 비공개면 코랩 왼쪽 열쇠(보안 비밀)에 GH_TOKEN 을 넣어 두세요
    from google.colab import userdata
    _tok = userdata.get("GH_TOKEN")
    REPO_URL = f"https://{_tok}@{REPO}"
except Exception:
    REPO_URL = f"https://{REPO}"
if not os.path.exists("/content/Cfinetune"):
    !git clone -q -b {BRANCH} {REPO_URL} /content/Cfinetune
%cd /content/Cfinetune
!git pull -q
%pip install -q -r requirements-colab.txt

In [ ]:
# 구글 드라이브: 결과(results/)를 세션이 끊겨도 남도록 저장
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, sys
DRIVE = "/content/drive/MyDrive/cfinetune"
os.makedirs(DRIVE, exist_ok=True)

def pull_results():
    if os.path.exists(f"{DRIVE}/results"):
        shutil.copytree(f"{DRIVE}/results", "results", dirs_exist_ok=True)

def push_results():
    shutil.copytree("results", f"{DRIVE}/results", dirs_exist_ok=True)
    print("드라이브에 결과 저장함")

pull_results()
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU 메모리 {mem:.0f}GB")
assert mem >= 38, "런타임 > 런타임 유형 변경에서 A100 를 고르세요"

In [ ]:
# 변환/양자화만 하면 되므로 CPU 빌드 (약 5분)
!LLAMA_BACKEND=cpu bash scripts/build_llama_cpp.sh /content/llama.cpp {DRIVE}/cache

In [ ]:
from _common import resolve_model_dir, load_config
cfg = load_config()
base = cfg["lora"]["base_variant"]
SRC = resolve_model_dir(cfg["model"]["id"])
!python scripts/02_prune.py --variant {base} --src {SRC} --out-dir /content/models/pruned

In [ ]:
OUT = f"/content/lora/{base}-lora"
!python scripts/04_train_lora.py --model /content/models/pruned/{base} --out {OUT}
shutil.copytree(f"{OUT}/adapter", f"{DRIVE}/lora/{base}-lora/adapter", dirs_exist_ok=True)
push_results()

In [ ]:
!python scripts/03_make_gguf.py --hf-dir {OUT}/merged --out /content/gguf/{base}-lora.gguf --quant MXFP4_MOE Q8_0
os.makedirs(f"{DRIVE}/gguf", exist_ok=True)
for q in ("MXFP4_MOE", "Q8_0"):   # 드라이브 용량이 부족하면 Q8_0 은 빼세요 (약 12GB)
    shutil.copy(f"/content/gguf/{base}-lora-{q}.gguf", f"{DRIVE}/gguf/")
!ls -la {DRIVE}/gguf